<a href="https://colab.research.google.com/github/rohitxsingh23/AI-Based-Password-Strength-and-Guessability-Analysis/blob/main/password_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

print("Hello, our password project!")

print ("passowrd")

Hello, our password project!
passowrd


In [4]:
print(data["strength"].value_counts())
print(data.isnull().sum())

strength
1    496801
0     89702
2     83137
Name: count, dtype: int64
password    1
strength    0
dtype: int64


In [5]:
data = data.dropna()
data = data.drop_duplicates()
data["password"] = data["password"].astype(str)

print(data.shape)

(669639, 2)


In [6]:
data = data.dropna()
data = data.drop_duplicates()
data["password"] = data["password"].astype(str)

print(data.shape)

(669639, 2)


In [ ]:
import pandas as pd

data = pd.read_csv("Kaggle Dataset.csv", on_bad_lines="skip")
print(data.head(10))
print(data.shape)

           password  strength
0          kzde5577         1
1          kino3434         1
2         visi7k1yr         1
3          megzy123         1
4       lamborghin1         1
5  AVYq1lDE4MgAZfNt         2
6          u6c8vhow         1
7          v1118714         1
8      universe2908         1
9          as326159         1
(669640, 2)


In [7]:
print(data["strength"].value_counts())
print(data.isnull().sum())

strength
1    496801
0     89701
2     83137
Name: count, dtype: int64
password    0
strength    0
dtype: int64


In [8]:
data["length"] = data["password"].str.len()
data["lowercase"] = data["password"].str.count(r"[a-z]")
data["uppercase"] = data["password"].str.count(r"[A-Z]")
data["digits"] = data["password"].str.count(r"[0-9]")
data["symbols"] = data["password"].str.count(r"[^a-zA-Z0-9]")

print(data.head(10))

           password  strength  length  lowercase  uppercase  digits  symbols
0          kzde5577         1       8          4          0       4        0
1          kino3434         1       8          4          0       4        0
2         visi7k1yr         1       9          7          0       2        0
3          megzy123         1       8          5          0       3        0
4       lamborghin1         1      11         10          0       1        0
5  AVYq1lDE4MgAZfNt         2      16          5          9       2        0
6          u6c8vhow         1       8          6          0       2        0
7          v1118714         1       8          1          0       7        0
8      universe2908         1      12          8          0       4        0
9          as326159         1       8          2          0       6        0


In [9]:
print(data.groupby("strength")[["length", "lowercase", "uppercase", "digits", "symbols"]].mean())

             length  lowercase  uppercase    digits   symbols
strength                                                     
0          6.549604   4.673181   0.078238  1.774741  0.023445
1          9.618964   6.068683   0.081548  3.449599  0.019134
2         15.932497   6.771907   5.823460  3.100509  0.236621


In [10]:
data["unique_chars"] = data["password"].apply(lambda p: len(set(p)))

data["has_repeat"] = data["password"].str.contains(r"(.)\1\1").astype(int)

common = "123|234|345|456|567|678|789|abc|qwerty|asdf|password|love"
data["has_pattern"] = data["password"].str.lower().str.contains(common).astype(int)

print(data.head(10))


/tmp/ipykernel_587/1343430039.py:3: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  data["has_repeat"] = data["password"].str.contains(r"(.)\1\1").astype(int)


           password  strength  length  lowercase  uppercase  digits  symbols  \
0          kzde5577         1       8          4          0       4        0   
1          kino3434         1       8          4          0       4        0   
2         visi7k1yr         1       9          7          0       2        0   
3          megzy123         1       8          5          0       3        0   
4       lamborghin1         1      11         10          0       1        0   
5  AVYq1lDE4MgAZfNt         2      16          5          9       2        0   
6          u6c8vhow         1       8          6          0       2        0   
7          v1118714         1       8          1          0       7        0   
8      universe2908         1      12          8          0       4        0   
9          as326159         1       8          2          0       6        0   

   unique_chars  has_repeat  has_pattern  
0             6           0            0  
1             6           0      

In [11]:
from sklearn.model_selection import train_test_split

features = ["length", "lowercase", "uppercase", "digits", "symbols",
            "unique_chars", "has_repeat", "has_pattern"]

X = data[features]
y = data["strength"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(X_train.shape)
print(X_test.shape)


(535711, 8)
(133928, 8)


In [12]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

model = DecisionTreeClassifier(max_depth=10, random_state=42)

model.fit(X_train, y_train)

predictions = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, predictions))

Accuracy: 1.0


In [13]:
def add_features(df):
    df["length"] = df["password"].str.len()
    df["lowercase"] = df["password"].str.count(r"[a-z]")
    df["uppercase"] = df["password"].str.count(r"[A-Z]")
    df["digits"] = df["password"].str.count(r"[0-9]")
    df["symbols"] = df["password"].str.count(r"[^a-zA-Z0-9]")
    df["unique_chars"] = df["password"].apply(lambda p: len(set(p)))
    df["has_repeat"] = df["password"].str.contains(r"(.)\1\1").astype(int)
    df["has_pattern"] = df["password"].str.lower().str.contains(common).astype(int)
    return df

test = pd.DataFrame({"password": [
    "123456",
    "aaaaaaaa",
    "qwerty123",
    "Password123!",
    "Rohit@2026",
    "Xk#92lpQ!7mz"
]})

test = add_features(test)
test["predicted"] = model.predict(test[features])

print(test[["password", "predicted"]])

       password  predicted
0        123456          0
1      aaaaaaaa          1
2     qwerty123          1
3  Password123!          1
4    Rohit@2026          1
5  Xk#92lpQ!7mz          1


/tmp/ipykernel_587/1354323722.py:8: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df["has_repeat"] = df["password"].str.contains(r"(.)\1\1").astype(int)
